
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 태스크 2 - 실버 - 골드 테이블
이 노트북은 노트북의 지침에 따라 작업의 태스크 2에 사용됩니다. **DEWD00 - 04-Creating a Simple Databricks Job**

## 환경 준비 (Databricks Free Edition)

- **카탈로그** `db_catalog`, **스키마** `db_schema` 를 사용합니다.
- 스키마 `db_schema` 아래 **`myfiles`** 볼륨을 만들고, **`employees.csv`** 를 업로드합니다.
- (다른 노트북에서 안내한 것과 동일한 데이터 레이아웃을 사용합니다.)


In [0]:
%sql
USE CATALOG db_catalog;
USE SCHEMA db_schema;

## 환경 구성

1. 기본 카탈로그를 **getstarted**로 설정하고 스키마를 특정 스키마로 설정합니다. 그런 다음 사용 가능한 테이블을 보고 현재 스키마에 테이블이 없는지 확인합니다.

In [0]:
%python
# 카탈로그 및 스키마 설정
spark.sql("USE CATALOG db_catalog")
spark.sql("USE SCHEMA db_schema")

### SILVER
**목적:** 브론즈 테이블을 변환하고 실버 테이블을 만듭니다.

1. **current_employees_silver_job** 테이블에서 **current_employees_bronze_job**라는 테이블을 만듭니다. 

    테이블은 다음을 수행합니다.
    - **ID**, **FirstName**, **Country** 열을 선택합니다.
    - **Role** 열을 대문자로 변환합니다.
    - 두 개의 새 열을 추가합니다. **CurrentTimeStamp** 및 **CurrentDate**.

In [0]:
-- 데이터를 최종 실버 테이블에 병합하는 데 사용할 임시 뷰 만들기
CREATE OR REPLACE TABLE current_employees_silver_job AS 
SELECT 
  ID,
  FirstName,
  Country,
  upper(Role) as Role,                 -- 역할 열 대문자로 변경
  current_timestamp() as CurrentTimeStamp,    -- 현재 날짜/시간 가져오기
  date(CurrentTimeStamp) as CurrentDate       -- 날짜 확인하기
FROM current_employees_bronze_job;

### GOLD
**목표:** 실버 테이블을 집계하여 최종 골드 테이블을 만듭니다.

1. **temp_view_total_roles_job**라는 이름의 임시 뷰를 만들고 역할별로 총 직원 수를 집계합니다. 그런 다음 보기 결과를 표시합니다.

In [0]:
CREATE OR REPLACE TEMP VIEW temp_view_total_roles_job AS 
SELECT
  Role, 
  count(*) as TotalEmployees
FROM current_employees_silver_job
GROUP BY Role;

2. **total_roles_gold_job**라는 이름의 최종 골드 테이블을 지정된 열로 만듭니다.

In [0]:
CREATE TABLE IF NOT EXISTS total_roles_gold_job (
  Role STRING,
  TotalEmployees INT
);

3. 집계된 임시 뷰 **temp_view_total_roles_job**의 모든 행을 테이블 **total_roles_gold_job**에 삽입하고 테이블의 기존 데이터를 덮어씁니다. 이렇게 하면 테이블의 데이터를 덮어쓰지만 기존 스키마와 테이블 정의 및 속성은 유지됩니다.

    다음을 확인합니다.
    - **num_affected_rows** 는 *4*입니다.
    - **num_inserted_rows** 는 *4*입니다.

In [0]:
INSERT OVERWRITE TABLE total_roles_gold_job
SELECT * 
FROM temp_view_total_roles_job;

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>